In [ ]:
import glob
import os
import re

import dask.dataframe as dd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import date as dt_date
from matplotlib.patches import Patch
from pathlib import Path
from pyproj import Transformer

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]',
         '(1000, 5000]']

In [ ]:
sdc_files = glob.glob('./stage_duration_curves_by_site/*.parquet')
sdc_data = {}

for sdc_file in sdc_files:
    df = pd.read_parquet(sdc_file)
    gauge_id = df['site_id'].iloc[0]
    sdc_data[gauge_id] = {
        'exceedance': df['exceedance'].values,
        'values': df['stage'].values,
        'site_id': gauge_id
    }

In [ ]:
results_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = './figures_stage_duration_curve/'

# Load your saved quality data
swot_quality_df = pd.read_parquet(os.path.join(results_dir, 'swot_observations_with_quality.parquet'))

# Make quality binary
swot_quality_df.loc[:, 'binary_quality'] = 0
swot_quality_df.loc[swot_quality_df['quality_level'].isin(
    ['unfiltered', 'height_threshold', 'useable']
), 'binary_quality'] = 1

In [ ]:
import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score, precision_recall_curve
import pandas as pd

def find_quality_breakpoint_sdc(gauge_id, sdc_data, swot_quality_df):
    """
    For each candidate breakpoint, compute how well it separates quality classes.
    """
    
    gauge_quality = swot_quality_df[swot_quality_df['site_id'] == gauge_id].copy()
    if len(gauge_quality) == 0:
        return None
    
    observed_stages = gauge_quality['wse_gauge'].values
    quality_flags = gauge_quality['binary_quality'].values
    
    # CHECK: Do we have both quality classes?
    if len(np.unique(quality_flags)) < 2:
        print(f"Gauge {gauge_id}: Skipped - only one quality class present (n={len(gauge_quality)})")
        return None
    
    # FIRST: Check if stage itself is a good predictor
    # This is the AUC for using stage continuously
    overall_auc = roc_auc_score(quality_flags, observed_stages)
    
    print(f"Gauge {gauge_id}: Overall AUC (stage as continuous predictor) = {overall_auc:.3f}, n_obs={len(gauge_quality)}")
    
    # Get candidate breakpoints
    candidate_stages = np.percentile(observed_stages, np.linspace(5, 95, 50))
    candidate_stages = np.unique(candidate_stages)
    candidate_stages = candidate_stages[~np.isnan(candidate_stages)]
    
    results = []
    
    for stage_threshold in candidate_stages:
        # Create binary prediction at this threshold
        predictions = (observed_stages > stage_threshold).astype(int)
        
        # Skip if no variation
        if len(np.unique(predictions)) < 2:
            continue
        
        # Compute confusion matrix
        tn = np.sum((predictions == 0) & (quality_flags == 0))
        tp = np.sum((predictions == 1) & (quality_flags == 1))
        fn = np.sum((predictions == 0) & (quality_flags == 1))
        fp = np.sum((predictions == 1) & (quality_flags == 0))
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        balanced_accuracy = (sensitivity + specificity) / 2
        
        results.append({
            'stage_threshold': stage_threshold,
            'balanced_accuracy': balanced_accuracy,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'tp': tp,
            'tn': tn,
            'fp': fp,
            'fn': fn,
        })
    
    if len(results) == 0:
        print(f"Gauge {gauge_id}: No valid thresholds found")
        return None
    
    results_df = pd.DataFrame(results)
    
    # Find threshold that maximizes balanced accuracy
    optimal_idx = results_df['balanced_accuracy'].idxmax()
    
    return {
        'gauge_id': gauge_id,
        'overall_auc': overall_auc,
        'optimal_threshold': results_df.loc[optimal_idx, 'stage_threshold'],
        'optimal_balanced_accuracy': results_df.loc[optimal_idx, 'balanced_accuracy'],
        'optimal_sensitivity': results_df.loc[optimal_idx, 'sensitivity'],
        'optimal_specificity': results_df.loc[optimal_idx, 'specificity'],
        'tp': int(results_df.loc[optimal_idx, 'tp']),
        'tn': int(results_df.loc[optimal_idx, 'tn']),
        'fp': int(results_df.loc[optimal_idx, 'fp']),
        'fn': int(results_df.loc[optimal_idx, 'fn']),
        'n_observations': len(gauge_quality),
    }

# Analyze quality distribution across gauges
quality_summary = []

for gauge_id in sdc_data.keys():
    gauge_quality = swot_quality_df[swot_quality_df['site_id'] == gauge_id].copy()
    
    if len(gauge_quality) == 0:
        continue
    
    quality_flags = gauge_quality['binary_quality'].values
    n_good = np.sum(quality_flags)
    n_bad = np.sum(quality_flags == 0)
    n_total = len(gauge_quality)
    pct_good = 100 * n_good / n_total
    
    quality_summary.append({
        'gauge_id': gauge_id,
        'n_total': n_total,
        'n_good': n_good,
        'n_bad': n_bad,
        'pct_good': pct_good,
        'quality_type': 'all_good' if n_bad == 0 else ('all_bad' if n_good == 0 else 'mixed')
    })

quality_df = pd.DataFrame(quality_summary)

print("="*70)
print("QUALITY DISTRIBUTION ACROSS GAUGES")
print("="*70)
print(f"\nTotal gauges: {len(quality_df)}")
print(f"Gauges with mixed quality: {len(quality_df[quality_df['quality_type'] == 'mixed'])}")
print(f"Gauges that are ALWAYS GOOD: {len(quality_df[quality_df['quality_type'] == 'all_good'])}")
print(f"Gauges that are ALWAYS BAD: {len(quality_df[quality_df['quality_type'] == 'all_bad'])}")

print("\n" + "="*70)
print("ALWAYS GOOD GAUGES:")
print("="*70)
always_good = quality_df[quality_df['quality_type'] == 'all_good']
if len(always_good) > 0:
    print(always_good[['gauge_id', 'n_total']])
else:
    print("None")

print("\n" + "="*70)
print("ALWAYS BAD GAUGES:")
print("="*70)
always_bad = quality_df[quality_df['quality_type'] == 'all_bad']
if len(always_bad) > 0:
    print(always_bad[['gauge_id', 'n_total']])
else:
    print("None")

print("\n" + "="*70)
print("QUALITY STATISTICS (for mixed gauges only):")
print("="*70)
mixed = quality_df[quality_df['quality_type'] == 'mixed']
print(f"Mean % good observations: {mixed['pct_good'].mean():.1f}%")
print(f"Median % good observations: {mixed['pct_good'].median():.1f}%")
print(f"Min % good observations: {mixed['pct_good'].min():.1f}%")
print(f"Max % good observations: {mixed['pct_good'].max():.1f}%")

# Save summary
quality_df.to_csv(os.path.join(save_dir, 'quality_distribution_by_gauge.csv'), index=False)

# Now proceed with breakpoint finding (only for mixed quality gauges)
print("\n" + "="*70)
print("FINDING BREAKPOINTS (for mixed quality gauges only)")
print("="*70 + "\n")

breakpoint_results = {}

for gauge_id in mixed['gauge_id']:  # Only use gauges with mixed quality
    result = find_quality_breakpoint_sdc(gauge_id, sdc_data, swot_quality_df)
    if result is not None:
        breakpoint_results[gauge_id] = result
        print(f"  → Breakpoint = {result['optimal_threshold']:.2f} m, Balanced Accuracy = {result['optimal_balanced_accuracy']:.3f}, Overall AUC = {result['overall_auc']:.3f}\n")

# Convert to DataFrame for analysis
breakpoints_df = pd.DataFrame(breakpoint_results).T
print(f"\nSuccessfully found breakpoints for {len(breakpoints_df)} gauges")
breakpoints_df.to_csv(os.path.join(save_dir, 'quality_breakpoints_sdc.csv'))
breakpoints_df

for gauge_id in sdc_data.keys():
    result = find_quality_breakpoint_sdc(gauge_id, sdc_data, swot_quality_df)
    if result is not None:
        breakpoint_results[gauge_id] = result
        print(f"  → Breakpoint = {result['optimal_threshold']:.2f} m, Balanced Accuracy = {result['optimal_balanced_accuracy']:.3f}, Overall AUC = {result['overall_auc']:.3f}\n")

# Convert to DataFrame for analysis
breakpoints_df = pd.DataFrame(breakpoint_results).T
print(f"\nSuccessfully analyzed {len(breakpoints_df)} gauges out of {len(sdc_data)}")
# breakpoints_df.to_csv(os.path.join(save_dir, 'quality_breakpoints_sdc.csv'))
breakpoints_df

In [ ]:
# Check your AUC distribution
print("="*70)
print("AUC DISTRIBUTION")
print("="*70)
print(f"Mean AUC: {breakpoints_df['overall_auc'].mean():.3f}")
print(f"Median AUC: {breakpoints_df['overall_auc'].median():.3f}")
print(f"Min AUC: {breakpoints_df['overall_auc'].min():.3f}")
print(f"Max AUC: {breakpoints_df['overall_auc'].max():.3f}")
print(f"Std Dev: {breakpoints_df['overall_auc'].std():.3f}")

print("\n" + "="*70)
print("GAUGES AT DIFFERENT THRESHOLDS")
print("="*70)
for threshold in [0.55, 0.60, 0.65, 0.70, 0.75]:
    n_gauges = len(breakpoints_df[breakpoints_df['overall_auc'] >= threshold])
    pct = 100 * n_gauges / len(breakpoints_df)
    print(f"AUC >= {threshold}: {n_gauges:3d} gauges ({pct:5.1f}%)")

In [ ]:
# Use 0.55 instead
auc_threshold = 0.55

high_auc_gauges = breakpoints_df[breakpoints_df['overall_auc'] >= auc_threshold].copy()

print(f"Proceeding with {len(high_auc_gauges)} gauges at AUC >= {auc_threshold}")
print(f"Average AUC of gauges above threshold: {high_auc_gauges['overall_auc'].mean():.3f}")

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact
import pandas as pd
import numpy as np
from scipy import stats

# ===================================================================
# FIXED FUNCTION - Replace your old one with this
# ===================================================================

def hypothesis_test_breakpoint(gauge_id, sdc_data, swot_quality_df, stage_breakpoint):
    """
    Test if stage breakpoint significantly separates quality classes.
    
    H0: Quality is independent of stage (no real difference)
    H1: Quality depends on stage (real difference exists)
    """
    
    # Get quality data for this gauge
    gauge_quality = swot_quality_df[swot_quality_df['site_id'] == gauge_id].copy()
    
    if len(gauge_quality) == 0:
        return None
    
    observed_stages = gauge_quality['wse_gauge'].values
    quality_flags = gauge_quality['binary_quality'].values
    
    # Create binary stage variable
    stage_above_breakpoint = (observed_stages > stage_breakpoint).astype(int)
    
    # Extract counts directly from binary arrays
    n_below_bad = np.sum((stage_above_breakpoint == 0) & (quality_flags == 0))
    n_below_good = np.sum((stage_above_breakpoint == 0) & (quality_flags == 1))
    n_above_bad = np.sum((stage_above_breakpoint == 1) & (quality_flags == 0))
    n_above_good = np.sum((stage_above_breakpoint == 1) & (quality_flags == 1))
    
    contingency_2x2 = np.array([[n_below_bad, n_below_good],
                                 [n_above_bad, n_above_good]])
    
    # Chi-square test (or Fisher's exact for small samples)
    if np.min(contingency_2x2) < 5:
        odds_ratio, p_value = fisher_exact(contingency_2x2)
    else:
        chi2, p_value, dof, expected = chi2_contingency(contingency_2x2)
        odds_ratio = (n_below_bad * n_above_good) / (n_below_good * n_above_bad) if (n_below_good * n_above_bad) > 0 else np.inf
    
    # Proportions
    p_good_below = n_below_good / (n_below_bad + n_below_good) if (n_below_bad + n_below_good) > 0 else 0
    p_good_above = n_above_good / (n_above_bad + n_above_good) if (n_above_bad + n_above_good) > 0 else 0
    
    # Z-test
    p_pool = (n_below_good + n_above_good) / (n_below_bad + n_below_good + n_above_bad + n_above_good)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/(n_below_bad + n_below_good) + 1/(n_above_bad + n_above_good)))
    z_stat = (p_good_above - p_good_below) / se if se > 0 else 0
    p_value_z = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    
    # Effect sizes
    p1_squared = np.arcsin(np.sqrt(p_good_below))
    p2_squared = np.arcsin(np.sqrt(p_good_above))
    cohens_h = 2 * (p2_squared - p1_squared)
    
    relative_risk = p_good_above / p_good_below if p_good_below > 0 else np.inf
    nnt = 1 / abs(p_good_above - p_good_below) if abs(p_good_above - p_good_below) > 0 else np.inf
    
    # Diagnostic metrics
    sensitivity = n_above_good / (n_above_good + n_below_good) if (n_above_good + n_below_good) > 0 else 0
    specificity = n_below_bad / (n_below_bad + n_above_bad) if (n_below_bad + n_above_bad) > 0 else 0
    ppv = n_above_good / (n_above_good + n_above_bad) if (n_above_good + n_above_bad) > 0 else 0
    npv = n_below_bad / (n_below_bad + n_below_good) if (n_below_bad + n_below_good) > 0 else 0
    
    return {
        'gauge_id': gauge_id,
        'stage_breakpoint': stage_breakpoint,
        'contingency_table': {
            'below_bad': n_below_bad,
            'below_good': n_below_good,
            'above_bad': n_above_bad,
            'above_good': n_above_good,
            'total': n_below_bad + n_below_good + n_above_bad + n_above_good
        },
        'proportions': {
            'pct_good_below': 100 * p_good_below,
            'pct_good_above': 100 * p_good_above,
            'difference': 100 * (p_good_above - p_good_below)
        },
        'statistical_tests': {
            'chi2_or_fisher_p': p_value,
            'z_test_p': p_value_z,
            'chi2_or_fisher_significant': p_value < 0.05,
        },
        'effect_sizes': {
            'odds_ratio': odds_ratio,
            'relative_risk': relative_risk,
            'cohens_h': cohens_h,
            'nnt': nnt,
        },
        'diagnostic_metrics': {
            'sensitivity': sensitivity,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv,
        }
    }


# ===================================================================
# NOW RUN HYPOTHESIS TESTS
# ===================================================================

auc_threshold = 0.55
high_auc_results = {gid: result for gid, result in breakpoint_results.items() 
                    if result['overall_auc'] >= auc_threshold}

print(f"Testing {len(high_auc_results)} gauges with AUC >= {auc_threshold}\n")

hypothesis_results = {}

for gauge_id, result in high_auc_results.items():
    breakpoint = result['optimal_threshold']
    hyp_test = hypothesis_test_breakpoint(gauge_id, sdc_data, swot_quality_df, breakpoint)
    
    if hyp_test is not None:
        hypothesis_results[gauge_id] = hyp_test
        
        sig_marker = "***" if hyp_test['statistical_tests']['chi2_or_fisher_significant'] else ""
        print(f"\n{'='*60}")
        print(f"Gauge: {gauge_id}, Breakpoint: {breakpoint:.2f} m {sig_marker}")
        print(f"{'='*60}")
        print(f"Contingency Table:")
        print(f"  Below breakpoint - Bad: {hyp_test['contingency_table']['below_bad']:3d}, Good: {hyp_test['contingency_table']['below_good']:3d}")
        print(f"  Above breakpoint - Bad: {hyp_test['contingency_table']['above_bad']:3d}, Good: {hyp_test['contingency_table']['above_good']:3d}")
        print(f"\nProportions:")
        print(f"  Good quality below breakpoint: {hyp_test['proportions']['pct_good_below']:6.1f}%")
        print(f"  Good quality above breakpoint: {hyp_test['proportions']['pct_good_above']:6.1f}%")
        print(f"  Difference: {hyp_test['proportions']['difference']:+6.1f}%")
        print(f"\nStatistical Significance:")
        print(f"  p-value: {hyp_test['statistical_tests']['chi2_or_fisher_p']:.4f} {sig_marker}")
        print(f"\nEffect Sizes:")
        print(f"  Odds Ratio: {hyp_test['effect_sizes']['odds_ratio']:.2f}")
        print(f"  Relative Risk: {hyp_test['effect_sizes']['relative_risk']:.2f}x")
        print(f"  Cohen's h: {hyp_test['effect_sizes']['cohens_h']:.2f}")
        print(f"  NNT: {hyp_test['effect_sizes']['nnt']:.1f}" if hyp_test['effect_sizes']['nnt'] != np.inf else "  NNT: ∞")
        print(f"\nDiagnostic Performance:")
        print(f"  Sensitivity: {100*hyp_test['diagnostic_metrics']['sensitivity']:6.1f}%")
        print(f"  Specificity: {100*hyp_test['diagnostic_metrics']['specificity']:6.1f}%")
        print(f"  PPV: {100*hyp_test['diagnostic_metrics']['ppv']:6.1f}%")


# ===================================================================
# SAVE RESULTS
# ===================================================================

results_summary = []
for gauge_id, result in hypothesis_results.items():
    results_summary.append({
        'gauge_id': gauge_id,
        'stage_breakpoint': result['stage_breakpoint'],
        'n_total': result['contingency_table']['total'],
        'n_good': result['contingency_table']['above_good'] + result['contingency_table']['below_good'],
        'pct_good_below': result['proportions']['pct_good_below'],
        'pct_good_above': result['proportions']['pct_good_above'],
        'pct_difference': result['proportions']['difference'],
        'p_value': result['statistical_tests']['chi2_or_fisher_p'],
        'significant': result['statistical_tests']['chi2_or_fisher_significant'],
        'odds_ratio': result['effect_sizes']['odds_ratio'],
        'relative_risk': result['effect_sizes']['relative_risk'],
        'cohens_h': result['effect_sizes']['cohens_h'],
        'nnt': result['effect_sizes']['nnt'],
        'sensitivity': result['diagnostic_metrics']['sensitivity'],
        'specificity': result['diagnostic_metrics']['specificity'],
        'ppv': result['diagnostic_metrics']['ppv'],
        'npv': result['diagnostic_metrics']['npv'],
    })

summary_df = pd.DataFrame(results_summary)

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Total hypothesis tests: {len(summary_df)}")
print(f"Significant results (p < 0.05): {summary_df['significant'].sum()}")
print(f"Non-significant results: {(~summary_df['significant']).sum()}")

if summary_df['significant'].sum() > 0:
    sig_df = summary_df[summary_df['significant']]
    print(f"\nAmong significant results:")
    print(f"  Mean Odds Ratio: {sig_df['odds_ratio'].mean():.2f}")
    print(f"  Mean Relative Risk: {sig_df['relative_risk'].mean():.2f}x")
    print(f"  Mean Cohen's h: {sig_df['cohens_h'].mean():.2f}")

summary_df.to_csv(os.path.join(save_dir, 'hypothesis_test_results.csv'), index=False)
print(f"\nSaved to: {os.path.join(save_dir, 'hypothesis_test_results.csv')}")

with open(os.path.join(save_dir, 'hypothesis_test_detailed_results.txt'), 'w') as f:
    for gauge_id, result in hypothesis_results.items():
        f.write(f"\n{'='*70}\n")
        f.write(f"Gauge: {gauge_id}\n")
        f.write(f"{'='*70}\n")
        f.write(f"Stage Breakpoint: {result['stage_breakpoint']:.2f} m\n")
        f.write(f"Total Observations: {result['contingency_table']['total']}\n")
        f.write(f"\nContingency Table:\n")
        f.write(f"  Below {result['stage_breakpoint']:.2f} m - Bad: {result['contingency_table']['below_bad']}, Good: {result['contingency_table']['below_good']}\n")
        f.write(f"  Above {result['stage_breakpoint']:.2f} m - Bad: {result['contingency_table']['above_bad']}, Good: {result['contingency_table']['above_good']}\n")
        f.write(f"\nQuality Proportions:\n")
        f.write(f"  Below breakpoint: {result['proportions']['pct_good_below']:.1f}% good\n")
        f.write(f"  Above breakpoint: {result['proportions']['pct_good_above']:.1f}% good\n")
        f.write(f"  Difference: {result['proportions']['difference']:+.1f}%\n")
        f.write(f"\np-value: {result['statistical_tests']['chi2_or_fisher_p']:.4f}\n")
        f.write(f"Significant: {result['statistical_tests']['chi2_or_fisher_significant']}\n")

print(f"Saved detailed results to: {os.path.join(save_dir, 'hypothesis_test_detailed_results.txt')}")